In [ ]:
# Authentication: run `earthengine authenticate` once and set the environment
# variable EE_PROJECT to your own Google Cloud project before executing.
import os
# =====================================================
# MANN-KENDALL + SEN'S SLOPE POR MUNICÍPIO (COLAB + GEE)
# =====================================================
# Fluxo:
# 1) Lê o ImageCollection do asset com as imagens anuais
# 2) Identifica os municípios únicos via propriedade NM_MUN
# 3) Para cada município:
#      - filtra as imagens do município
#      - usa a propriedade ANO para ordenar a série temporal
#      - calcula tendências pixel a pixel para cada banda:
#           S, varS, Z, tau, p_value, sen_slope, sen_intercept
#           mean, median, stdDev
#      - exporta 1 imagem por município contendo todas as bandas de trend
#
# Saída:
#   projects/YOUR_PROJECT/assets/RSEI_TRENDS_2026_new/<nome_resultado>
#
# Observações:
# - Var(S) implementada sem correção de empates (ties)
# - Sen slope fica em "unidade da banda por ano"
# =====================================================

# !pip -q install earthengine-api geemap

import math
import re
import unicodedata
import ee
import geemap

# ================================
# 1. Autenticação
# ================================
try:
    ee.Initialize(project=os.environ.get('EE_PROJECT'))
except Exception:
    ee.Authenticate()
    ee.Initialize(project=os.environ.get('EE_PROJECT'))



In [ ]:
# -----------------------------
# PARÂMETROS
# -----------------------------
ASSET_IDS_LEITURA = [
    'projects/YOUR_PROJECT/assets/RSEI_Bands_2026_new_new',
    'projects/YOUR_PROJECT_2/assets/RSEI_Bands_2026_new_new'
]

OUTPUT_ASSET_FOLDER = "projects/YOUR_PROJECT_2/assets/RSEI_TRENDS_2026_new_new"

MUNIC_FIELD = "NM_MUN"
YEAR_FIELD = "ANO"

YEAR_START = 1985
YEAR_END = 2025

BANDS_TO_ANALYZE = ["RSEI"]   # pode ampliar depois, ex.: ["RSEI", "NDVI", "WET", ...]
MAX_PIXELS = 1e13

# -----------------------------
# HELPERS DE LEITURA DOS ASSETS
# -----------------------------
def listar_imagens_do_asset(asset_id):
    """
    Tenta ler o asset diretamente como ImageCollection.
    Se falhar, tenta listar os filhos IMAGE via ee.data.listAssets.
    """
    try:
        ic = ee.ImageCollection(asset_id)
        _ = ic.size().getInfo()  # força leitura
        print(f"Asset lido como ImageCollection: {asset_id}")
        return ic
    except Exception as e:
        print(f"Falha ao ler como ImageCollection: {asset_id}")
        print(f"Motivo: {e}")
        print("Tentando via ee.data.listAssets...")

        resp = ee.data.listAssets({'parent': asset_id})
        assets = resp.get('assets', [])
        image_ids = [a['id'] for a in assets if a.get('type') == 'IMAGE']

        if not image_ids:
            print(f"Nenhuma imagem encontrada em: {asset_id}")
            return ee.ImageCollection([])

        print(f"{len(image_ids)} imagens encontradas em {asset_id} via listAssets.")
        return ee.ImageCollection.fromImages([ee.Image(i) for i in image_ids])


def padronizar_imagem(img, asset_origem):
    """
    Garante propriedades mínimas para merge e ordenação.
    """
    ano_raw = img.get(YEAR_FIELD)
    ano_num = ee.Number.parse(ee.String(ano_raw))

    nm = ee.String(ee.Algorithms.If(img.get(MUNIC_FIELD), img.get(MUNIC_FIELD), ''))
    return (ee.Image(img)
            .set('asset_origem', asset_origem)
            .set('year_num', ano_num)
            .set(YEAR_FIELD, ano_num)
            .set(MUNIC_FIELD, nm)
            .set('system_time_start', ee.Date.fromYMD(ano_num, 1, 1).millis()))


def carregar_e_unir_colecoes(asset_ids):
    """
    Lê, padroniza e junta as coleções dos múltiplos assets.
    Mantém a lógica do script e apenas amplia a leitura para múltiplos roots.
    """
    colecao_unida = ee.ImageCollection([])

    for asset_id in asset_ids:
        ic = listar_imagens_do_asset(asset_id)
        ic = ic.map(lambda img: padronizar_imagem(img, asset_id))
        colecao_unida = colecao_unida.merge(ic)

    # filtro explícito do intervalo solicitado
    colecao_unida = (colecao_unida
                     .filter(ee.Filter.gte(YEAR_FIELD, YEAR_START))
                     .filter(ee.Filter.lte(YEAR_FIELD, YEAR_END)))

    # ordenação global por ANO; a ordenação por município é reforçada na execução
    colecao_unida = colecao_unida.sort(YEAR_FIELD)

    return colecao_unida


# -----------------------------
# IMAGECOLLECTION DE ENTRADA
# -----------------------------
col = carregar_e_unir_colecoes(ASSET_IDS_LEITURA)
print("Total de imagens na coleção unida:", col.size().getInfo())


In [ ]:
# =====================================================
# FUNÇÕES AUXILIARES
# =====================================================

def sanitize_name(text):
    """
    Converte string em nome seguro para asset/task:
    - remove acentos
    - troca espaços por underscore
    - remove caracteres especiais
    """
    text = str(text).strip()
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9]+', '_', text)
    text = re.sub(r'_+', '_', text).strip('_')
    return text


def ensure_asset_folder(asset_path):
    """
    Cria a pasta de saída se ela não existir.
    """
    try:
        ee.data.getAsset(asset_path)
        print(f"Pasta de saída já existe: {asset_path}")
    except Exception:
        parent = '/'.join(asset_path.split('/')[:-1])
        print(f"Criando pasta de saída: {asset_path}")
        ee.data.createAsset({'type': 'FOLDER'}, asset_path)


def normal_cdf_from_z(z_img):
    """
    CDF da Normal padrão usando erf:
    Phi(z) = 0.5 * (1 + erf(z / sqrt(2)))
    """
    return z_img.divide(math.sqrt(2)).erf().add(1).multiply(0.5)


def rename_metric_bands(bands, metric):
    """
    Ex.: ['RSEI'] + 'mean' -> ['RSEI_mean']
    """
    return [f"{b}_{metric}" for b in bands]


def validate_required_fields_and_bands(ic, bands_to_analyze, municip_field, year_field):
    """
    Checa campos e bandas na primeira imagem.
    """
    first = ee.Image(ic.first())
    if first is None:
        raise ValueError("A coleção está vazia.")

    props = first.propertyNames().getInfo()
    band_names = first.bandNames().getInfo()

    print("Propriedades encontradas na primeira imagem:")
    print(props)
    print("\nBandas encontradas na primeira imagem:")
    print(band_names)

    if municip_field not in props:
        raise ValueError(f"Propriedade '{municip_field}' não encontrada na coleção.")
    if year_field not in props:
        raise ValueError(f"Propriedade '{year_field}' não encontrada na coleção.")

    missing_bands = [b for b in bands_to_analyze if b not in band_names]
    if missing_bands:
        raise ValueError(f"As seguintes bandas não existem na coleção: {missing_bands}")

    return band_names, props


def add_numeric_year_property(img):
    """
    Garante que a propriedade ANO fique como número.
    """
    year_num = ee.Number.parse(ee.String(img.get(YEAR_FIELD)))
    return img.set('year_num', year_num)


def mk_sen_for_collection(ic, bands):
    """
    Calcula Mann-Kendall + Sen's slope para um ImageCollection ordenado por ano.
    Retorna uma única imagem com bandas:
      band_S, band_varS, band_Z, band_tau, band_p_value,
      band_sen_slope, band_sen_intercept,
      band_mean, band_median, band_stdDev
    """
    ic = ic.map(add_numeric_year_property).sort('year_num')
    n = ic.size()

    # Estatísticas descritivas da série original
    mean_img = ic.select(bands).mean().rename(rename_metric_bands(bands, "mean"))
    median_img = ic.select(bands).median().rename(rename_metric_bands(bands, "median"))
    std_img = ic.select(bands).reduce(ee.Reducer.stdDev()).rename(rename_metric_bands(bands, "stdDev"))

    # Se n < 2, não é possível calcular tendência
    # Ainda assim retornamos estatísticas + bandas de tendência zeradas
    n_info = n.getInfo()
    if n_info < 2:
        print("Série com menos de 2 imagens. Tendência não pode ser calculada; serão geradas bandas zeradas.")
        first = ee.Image(ic.first()).select(bands)
        zero_base = first.multiply(0)

        S_img = zero_base.rename(rename_metric_bands(bands, "S"))
        varS_img = zero_base.rename(rename_metric_bands(bands, "varS"))
        Z_img = zero_base.rename(rename_metric_bands(bands, "Z"))
        tau_img = zero_base.rename(rename_metric_bands(bands, "tau"))
        p_img = zero_base.rename(rename_metric_bands(bands, "p_value"))
        slope_img = zero_base.rename(rename_metric_bands(bands, "sen_slope"))
        intercept_img = zero_base.rename(rename_metric_bands(bands, "sen_intercept"))

        return ee.Image.cat([
            S_img, varS_img, Z_img, tau_img, p_img,
            slope_img, intercept_img,
            mean_img, median_img, std_img
        ])

    # Lista ordenada para construir comparações pareadas
    img_list = ic.toList(n)

    sign_images = []
    slope_images = []

    for i in range(n_info - 1):
        img_i = ee.Image(img_list.get(i)).select(bands)
        year_i = ee.Number(ee.Image(img_list.get(i)).get('year_num'))

        for j in range(i + 1, n_info):
            img_j = ee.Image(img_list.get(j)).select(bands)
            year_j = ee.Number(ee.Image(img_list.get(j)).get('year_num'))

            dy = img_j.subtract(img_i)
            dt = year_j.subtract(year_i)

            # sign(dy): 1 se >0, -1 se <0, 0 se igual
            sign_ij = dy.gt(0).subtract(dy.lt(0)).toFloat()
            slope_ij = dy.divide(dt).toFloat()

            sign_images.append(sign_ij)
            slope_images.append(slope_ij)

    # Soma dos sinais -> S
    S_raw = ee.ImageCollection.fromImages(sign_images).sum()
    S_img = S_raw.rename(rename_metric_bands(bands, "S"))

    # Variância de S sem correção de empates:
    # var(S) = n(n-1)(2n+5)/18
    n_num = ee.Number(n)
    varS_scalar = n_num.multiply(n_num.subtract(1)).multiply(n_num.multiply(2).add(5)).divide(18)
    varS_raw = ee.Image.constant(varS_scalar).toFloat().rename(bands)
    varS_img = varS_raw.rename(rename_metric_bands(bands, "varS"))

    # Z
    sqrt_varS = varS_raw.sqrt()

    z_pos = S_raw.subtract(1).divide(sqrt_varS)
    z_neg = S_raw.add(1).divide(sqrt_varS)
    z_zero = ee.Image.constant(0).rename(bands).toFloat()

    Z_raw = ee.Image(ee.Algorithms.If(
        True,
        z_zero.where(S_raw.gt(0), z_pos).where(S_raw.lt(0), z_neg),
        z_zero
    )).toFloat()

    Z_img = Z_raw.rename(rename_metric_bands(bands, "Z"))

    # Kendall tau
    n_pairs = n_num.multiply(n_num.subtract(1)).divide(2)
    tau_raw = S_raw.divide(n_pairs).toFloat()
    tau_img = tau_raw.rename(rename_metric_bands(bands, "tau"))

    # p-value bicaudal aproximado
    absZ = Z_raw.abs()
    p_raw = ee.Image.constant(1).subtract(normal_cdf_from_z(absZ)).multiply(2).toFloat()
    p_img = p_raw.rename(rename_metric_bands(bands, "p_value"))

    # Sen's slope = mediana das inclinações pareadas
    slope_raw = ee.ImageCollection.fromImages(slope_images).median().toFloat()
    slope_img = slope_raw.rename(rename_metric_bands(bands, "sen_slope"))

    # Sen intercept = mediana(y_i - slope * ano_i)
    intercept_ic = ic.map(
        lambda im: ee.Image(im).select(bands).subtract(
            slope_raw.multiply(ee.Number(im.get('year_num')))
        )
    )
    intercept_raw = intercept_ic.median().toFloat()
    intercept_img = intercept_raw.rename(rename_metric_bands(bands, "sen_intercept"))

    # Empilha todas as bandas
    out = ee.Image.cat([
        S_img,
        varS_img,
        Z_img,
        tau_img,
        p_img,
        slope_img,
        intercept_img,
        mean_img,
        median_img,
        std_img
    ])

    return out


def export_municipality_trend_image(ic_mun, mun_name, bands):
    """
    Calcula a imagem de tendência do município e exporta para Asset.
    """
    ic_mun = ic_mun.map(add_numeric_year_property).sort('year_num')

    count_imgs = ic_mun.size().getInfo()
    print(f"  Número de imagens para o município: {count_imgs}")

    if count_imgs == 0:
        print("  Município sem imagens. Exportação ignorada.")
        return None

    # Imagem final com todas as bandas de trend
    trend_img = mk_sen_for_collection(ic_mun, bands)

    # Metadados
    years = ic_mun.aggregate_array(YEAR_FIELD).getInfo()
    years_sorted = sorted([int(y) for y in years])

    safe_name = sanitize_name(mun_name)
    asset_name = f"trend_{safe_name}"
    asset_id = f"{OUTPUT_ASSET_FOLDER}/{asset_name}"

    trend_img = trend_img.set({
        'municipio': mun_name,
        'municipio_sanitizado': safe_name,
        'n_images': count_imgs,
        'years': years_sorted,
        'source_asset': ' | '.join(ASSET_IDS_LEITURA)
    })

    # Região, escala e CRS a partir da primeira imagem
    first_img = ee.Image(ic_mun.first()).select(bands[0])
    proj = first_img.projection()
    scale = proj.nominalScale().getInfo()
    crs = proj.crs().getInfo()
    region = ic_mun.geometry()

    print(f"  Exportando para: {asset_id}")
    print(f"  Anos: {years_sorted}")
    print(f"  Escala: {scale}")
    print(f"  CRS: {crs}")

    task = ee.batch.Export.image.toAsset(
        image=trend_img,
        description=asset_name[:100],
        assetId=asset_id,
        region=region,
        scale=scale,
        crs=crs,
        maxPixels=MAX_PIXELS
    )
    task.start()
    return task




In [ ]:
# =====================================================
# EXECUÇÃO
# =====================================================

# 1) Garante pasta de saída
ensure_asset_folder(OUTPUT_ASSET_FOLDER)

# 2) Validação
band_names, props = validate_required_fields_and_bands(
    col,
    BANDS_TO_ANALYZE,
    MUNIC_FIELD,
    YEAR_FIELD
)

# 3) Municípios únicos
municipios = col.aggregate_array(MUNIC_FIELD).distinct().sort().getInfo()
municipios = [m for m in municipios if m is not None and str(m).strip() != ""]

print("\nTotal de municípios encontrados:", len(municipios))
print("Municípios:")
for m in municipios:
    print(" -", m)

# 4) Exporta 1 imagem por município
tasks = []

for idx, mun in enumerate(municipios, start=1):
    print("\n" + "=" * 80)
    print(f"[{idx}/{len(municipios)}] Processando município: {mun}")

    # mantém a lógica do script, mas agora a coleção vem dos dois assets
    # e a ordenação é garantida por município + ANO (1985 a 2025)
    ic_mun = (col
              .filter(ee.Filter.eq(MUNIC_FIELD, mun))
              .filter(ee.Filter.gte(YEAR_FIELD, YEAR_START))
              .filter(ee.Filter.lte(YEAR_FIELD, YEAR_END))
              .sort(YEAR_FIELD))

    try:
        anos = ic_mun.aggregate_array(YEAR_FIELD).getInfo()
        print(f"  Série temporal (anos): {anos}")

        task = export_municipality_trend_image(ic_mun, mun, BANDS_TO_ANALYZE)
        if task is not None:
            tasks.append((mun, task))
    except Exception as e:
        print(f"  Erro ao processar {mun}: {e}")

print("\n" + "=" * 80)
print("Total de tasks iniciadas:", len(tasks))
for mun, task in tasks:
    print(f"- {mun}: {task.status().get('state')}")

print("\nProcessamento concluído. Verifique as tasks no Earth Engine/Tasks.")
